# Análise da Rede de E-mails da Enron  
## T1 — Caracterização de Grafos de Escala Livre com Dados do SNAP

Este notebook reorganiza e explica melhor o projeto da equipe, **sem remover as informações já presentes na versão anterior**.  
A proposta é manter a mesma análise, mas com uma narrativa mais didática para leitura, reprodução e apresentação oral.


## 1. Contexto do trabalho

A atividade pede a caracterização estrutural de uma rede real do SNAP e a investigação da hipótese de comportamento de escala livre.

Neste projeto, foi escolhido o dataset **Enron Email Network**, da categoria **Communication Network**.

### O que queremos responder?
- Como modelar essa base como grafo?
- Quais são suas métricas estruturais básicas?
- Como é a distribuição de graus da rede?
- Existe evidência de cauda pesada?
- A rede pode ser considerada de escala livre? Em que medida?

### Fonte do dataset
- SNAP — Stanford Large Network Dataset Collection  
- Dataset: **Enron Email Network**  
- Página: https://snap.stanford.edu/data/email-Enron.html


## 2. Definição formal do grafo

A modelagem adotada neste trabalho foi:

- **grafo não direcionado**;
- **grafo não ponderado**;
- representado por **listas de adjacência**;
- com uso de **Bag**, no estilo `algs4`.

### Interpretação
- cada vértice representa um endereço de e-mail;
- cada aresta representa uma conexão entre dois endereços;
- como a própria descrição do SNAP trata a rede como **undirected**, a análise foi mantida nesse formato.

### Observação importante
O uso de `Bag` foi mantido porque conversa bem com a proposta da disciplina.  
Porém, como `Bag` **não impede duplicatas**, o tratamento correto das arestas precisou ser feito no carregamento do arquivo.


In [ ]:
import gzip
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import powerlaw
from algs4.bag import Bag


## 3. Classe `Graph`

A estrutura abaixo representa um grafo simples não direcionado com listas de adjacência.

### Ideia central
Se existe uma aresta `(v, w)`:
- `w` entra na adjacência de `v`;
- `v` entra na adjacência de `w`.

Também incluímos métodos para:
- grau de um vértice;
- densidade;
- grau médio;
- clustering local;
- clustering médio;
- obtenção do vetor de graus.


In [ ]:
class Graph:
    """
    Grafo não direcionado representado por listas de adjacência com Bag.

    Cada vértice representa um nó.
    Cada aresta (v, w) representa uma conexão entre v e w.

    Como o grafo é não direcionado:
    - w entra na adjacência de v
    - v entra na adjacência de w

    Observação:
    - Bag não impede duplicatas
    - por isso, o tratamento de duplicatas será feito na leitura do arquivo
    """

    def __init__(self, V):
        self._V = V
        self._E = 0
        self.adj = [Bag() for _ in range(V)]

    def V(self):
        return self._V

    def E(self):
        return self._E

    def add_edge(self, v, w):
        """Adiciona uma aresta não direcionada entre v e w."""
        self.adj[v].add(w)
        self.adj[w].add(v)
        self._E += 1

    def degree(self, v):
        """Retorna o grau do vértice v."""
        count = 0
        for _ in self.adj[v]:
            count += 1
        return count

    def neighbors(self, v):
        """Retorna a lista ordenada de vizinhos de v."""
        return sorted(list(self.adj[v]))

    def neighbors_set(self, v):
        """Retorna os vizinhos de v em formato set."""
        return set(self.adj[v])

    def degrees_array(self, remove_zeros=False):
        """Retorna um array com os graus de todos os vértices."""
        deg = np.array([self.degree(v) for v in range(self._V)], dtype=int)
        if remove_zeros:
            return deg[deg > 0]
        return deg

    def density(self):
        """
        Densidade de um grafo não direcionado simples:
            D = 2E / (V(V-1))
        """
        if self._V <= 1:
            return 0.0
        return (2 * self._E) / (self._V * (self._V - 1))

    def avg_degree(self):
        """
        Grau médio de um grafo não direcionado:
            grau_medio = 2E / V
        """
        if self._V == 0:
            return 0.0
        return (2 * self._E) / self._V

    def local_clustering(self, v):
        """
        Coeficiente de clustering local do vértice v:
            C(v) = 2 * número de ligações entre vizinhos / (k * (k - 1))
        Se grau < 2, retorna 0.
        """
        neighbors = list(self.adj[v])
        k = len(neighbors)

        if k < 2:
            return 0.0

        links_between_neighbors = 0

        for i in range(k):
            u = neighbors[i]
            for j in range(i + 1, k):
                w = neighbors[j]
                if w in self.neighbors_set(u):
                    links_between_neighbors += 1

        return (2 * links_between_neighbors) / (k * (k - 1))

    def average_clustering(self):
        """Coeficiente de clustering médio do grafo."""
        if self._V == 0:
            return 0.0

        values = [self.local_clustering(v) for v in range(self._V)]
        return float(np.mean(values))


## 4. Carregamento do dataset e tratamento da base

Aqui está um dos pontos mais importantes do projeto.

### Problema encontrado na base
O arquivo contém muitos pares repetidos em ordem invertida, por exemplo:
- `1 0`
- `0 1`

Como o grafo é **não direcionado**, essas duas linhas representam a **mesma aresta**.

Isso bate com a descrição do SNAP:

> "Nodes of the network are email addresses and if an address i sent at least one email to address j, the graph contains an undirected edge from i to j."

E também faz sentido quando comparamos com os dados oficiais da base:
- **Nodes = 36692**
- **Edges = 183831**

Sem o tratamento, o número de arestas pode aparecer aproximadamente dobrado.

### Estratégia usada
Cada aresta `(v, w)` é convertida para:
- `(min(v, w), max(v, w))`

Depois disso, usamos um `set` para garantir unicidade.

Além disso:
- linhas de comentário são ignoradas;
- linhas vazias são ignoradas;
- auto-laços são descartados.


In [ ]:
def load_graph_from_gz(filename):
    """
    Lê o arquivo .gz e monta o grafo não direcionado com Bag.

    Como o dataset da Enron traz as conexões nos dois sentidos:
        u v
        v u

    fazemos o tratamento de duplicatas na leitura, usando uma forma
    canônica da aresta:
        (min(u, v), max(u, v))

    Assim, cada aresta não direcionada entra apenas uma vez.
    """
    edges = []
    seen = set()
    max_node = 0

    with gzip.open(filename, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            v, w = map(int, line.split())

            # ignora auto-laços
            if v == w:
                continue

            # normaliza a aresta para não direcionado
            edge = (min(v, w), max(v, w))

            if edge not in seen:
                seen.add(edge)
                edges.append(edge)
                max_node = max(max_node, v, w)

    g = Graph(max_node + 1)

    for v, w in edges:
        g.add_edge(v, w)

    return g


## 5. Funções auxiliares para métricas e inspeção

Essas funções ajudam a:
- imprimir métricas básicas;
- verificar o lema do aperto de mãos;
- resumir os graus;
- inspecionar manualmente alguns vértices para explicar o comportamento da rede na apresentação.


In [ ]:
def print_basic_metrics(g):
    """Imprime as métricas básicas do grafo."""
    degrees_all = g.degrees_array(remove_zeros=False)
    degrees_nonzero = g.degrees_array(remove_zeros=True)

    print("=" * 60)
    print("MÉTRICAS BÁSICAS DO GRAFO")
    print("=" * 60)
    print("Modelagem adotada: grafo não direcionado e não ponderado")
    print(f"|V| (número de vértices): {g.V()}")
    print(f"|E| (número de arestas): {g.E()}")
    print(f"Densidade: {g.density():.8f}")
    print(f"Grau médio: {g.avg_degree():.4f}")
    print(f"Clustering médio: {g.average_clustering():.6f}")
    print()

    soma_graus = degrees_all.sum()
    print(f"Soma dos graus: {soma_graus}")
    print(f"Verificação do lema do aperto de mãos: 2|E| = {2 * g.E()}")
    print()

    print("Resumo dos graus (todos os vértices):")
    print(f"  Mínimo:  {degrees_all.min()}")
    print(f"  Mediana: {int(np.median(degrees_all))}")
    print(f"  Média:   {degrees_all.mean():.4f}")
    print(f"  Máximo:  {degrees_all.max()}")
    print()

    print("Resumo dos graus (apenas graus > 0):")
    print(f"  Quantidade de vértices conectados: {len(degrees_nonzero)}")
    print(f"  Mínimo:  {degrees_nonzero.min()}")
    print(f"  Mediana: {int(np.median(degrees_nonzero))}")
    print(f"  Média:   {degrees_nonzero.mean():.4f}")
    print(f"  Máximo:  {degrees_nonzero.max()}")
    print()


def inspect_vertices(g, vertices):
    """
    Inspeção manual de alguns vértices.
    Útil para validar a modelagem e explicar na apresentação.
    """
    print("=" * 60)
    print("INSPEÇÃO MANUAL DE VÉRTICES")
    print("=" * 60)

    for v in vertices:
        if v < 0 or v >= g.V():
            print(f"Vértice {v}: fora do intervalo válido.")
            continue

        vizinhos = g.neighbors(v)
        print(f"Vértice {v}")
        print(f"  Grau: {g.degree(v)}")
        print(f"  Clustering local: {g.local_clustering(v):.6f}")
        print(f"  Vizinhos (até 15): {vizinhos[:15]}")
        print()


## 6. Distribuição de graus `P(k)`

A distribuição de graus responde à pergunta:

> Qual é a probabilidade de um vértice ter grau `k`?

Ela é central neste trabalho porque grafos de escala livre costumam apresentar:
- muitos vértices com baixo grau;
- poucos vértices com grau muito alto;
- uma cauda pesada que, em certos casos, pode ser compatível com lei de potência.


In [ ]:
def degree_distribution_pk(degrees):
    """
    Retorna a distribuição de graus P(k), isto é,
    a probabilidade de um vértice ter grau k.
    """
    freq = Counter(degrees)
    total = len(degrees)

    k_vals = np.array(sorted(freq.keys()), dtype=int)
    pk_vals = np.array([freq[k] / total for k in k_vals], dtype=float)

    return k_vals, pk_vals


def plot_degree_distribution_linear(degrees, title, save_path=None):
    """
    Plota a distribuição de graus P(k) em escala linear.
    Eixo X: grau k
    Eixo Y: P(k) = frequência relativa
    """
    k_vals, pk_vals = degree_distribution_pk(degrees)

    plt.figure(figsize=(10, 6))
    plt.bar(k_vals, pk_vals, width=0.9, alpha=0.85, edgecolor="black")

    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlabel("Grau (k)")
    plt.ylabel("P(k)")
    plt.grid(alpha=0.25, axis="y")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


def plot_loglog(degrees, title, save_path=None):
    """
    Plota a distribuição de graus P(k) em escala log-log.
    Eixo X: grau k
    Eixo Y: P(k)
    Ambos em escala logarítmica.
    """
    k_vals, pk_vals = degree_distribution_pk(degrees)

    plt.figure(figsize=(10, 6))
    plt.scatter(k_vals, pk_vals, s=20, alpha=0.85, label="P(k) observada")

    plt.xscale("log")
    plt.yscale("log")

    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlabel("Grau (k) [escala log]")
    plt.ylabel("P(k) [escala log]")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


## 7. Ajuste de lei de potência

Aqui usamos a biblioteca `powerlaw` para ajustar uma lei de potência na distribuição de graus positivos.

### Parâmetros reportados
- `gamma`: expoente estimado;
- `xmin`: ponto a partir do qual a cauda passa a ser analisada;
- `sigma`: erro padrão da estimativa;
- `KS`: distância Kolmogorov-Smirnov do ajuste;
- `n_cauda`: quantidade de observações na cauda.

### Ponto metodológico importante
Este notebook **não conclui automaticamente** que a rede inteira é escala livre apenas porque há um ajuste na cauda.  
A interpretação correta é mais cuidadosa: existe uma **cauda que pode ser analisada por power law**, e isso deve ser discutido criticamente.


In [ ]:
def fit_powerlaw(degrees):
    """
    Ajusta uma lei de potência aos graus positivos.

    Retorna:
    - fit
    - gamma
    - xmin
    - sigma
    - ks
    - n_tail
    """
    fit = powerlaw.Fit(degrees, discrete=True, verbose=False)

    gamma = fit.power_law.alpha
    xmin = fit.power_law.xmin
    sigma = fit.power_law.sigma

    ks = getattr(fit.power_law, "D", None)
    if ks is None:
        ks = fit.power_law.KS()

    n_tail = int(np.sum(np.array(degrees) >= xmin))

    return fit, gamma, xmin, sigma, ks, n_tail


def plot_loglog_with_fit(degrees, fit, ks, n_tail, title, save_path=None):
    """Plota P(k) em log-log com linha visual do ajuste de power law."""
    k_vals, pk_vals = degree_distribution_pk(degrees)

    gamma = fit.power_law.alpha
    xmin = fit.power_law.xmin
    sigma = fit.power_law.sigma

    mask = k_vals >= xmin
    k_sel = k_vals[mask]
    pk_sel = pk_vals[mask]

    plt.figure(figsize=(10, 6))
    plt.scatter(k_vals, pk_vals, s=20, alpha=0.85, label="P(k) empírica")

    plt.xscale("log")
    plt.yscale("log")
    plt.grid(alpha=0.25)

    if len(k_sel) >= 2:
        k0 = float(k_sel[0])
        p0 = float(pk_sel[0])

        C = p0 * (k0 ** gamma)

        k_line = np.linspace(k_sel.min(), k_sel.max(), 400)
        pk_line = C * (k_line ** (-gamma))

        plt.plot(
            k_line,
            pk_line,
            linewidth=2.2,
            label=f"Ajuste visual power law (γ = {gamma:.3f})",
        )

        plt.axvline(
            xmin,
            linestyle="--",
            linewidth=1.5,
            label=f"xmin = {xmin}",
        )

    info_text = (
        f"γ = {gamma:.3f}\n"
        f"xmin = {xmin}\n"
        f"sigma = {sigma:.3f}\n"
        f"KS = {ks:.4f}\n"
        f"n_cauda = {n_tail}"
    )

    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlabel("Grau (k)")
    plt.ylabel("P(k)")

    plt.text(
        0.97,
        0.97,
        info_text,
        transform=plt.gca().transAxes,
        fontsize=10,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(boxstyle="round", alpha=0.15),
    )

    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


def print_scale_free_conclusion(gamma, xmin, ks, n_tail):
    """Imprime uma discussão inicial sobre o ajuste de power law."""
    print("=" * 60)
    print("DISCUSSÃO INICIAL SOBRE A LEI DE POTÊNCIA")
    print("=" * 60)
    print("A classificação como escala livre não deve ser feita apenas pela aparência do gráfico log-log.")
    print("É importante observar o ajuste estatístico e o tamanho da cauda.")
    print()

    print(f"γ estimado: {gamma:.4f}")
    print(f"xmin: {xmin}")
    print(f"KS: {ks:.4f}")
    print(f"n_cauda: {n_tail}")
    print()

    print("Conclusão inicial:")
    print("A distribuição apresenta uma cauda que pode ser analisada por meio de lei de potência.")
    print("Mesmo assim, isso não significa automaticamente que toda a rede seja perfeitamente escala livre.")
    print("A interpretação correta é dizer que há um ajuste de power law na cauda da distribuição de graus.")
    print()


## 8. Caminho do dataset

Na versão anterior do notebook, o nome do arquivo aparecia apenas como:

```python
nome_arquivo = "email-Enron.txt.gz"
```

Para a reprodução ficar mais robusta dentro da estrutura do repositório, aqui usamos o caminho relativo completo a partir da pasta `notebooks/`.


In [ ]:
dataset_path = Path("../data/email-Enron.txt.gz")
print(f"Abrindo arquivo: {dataset_path}...")

g = load_graph_from_gz(dataset_path)

degrees_all = g.degrees_array(remove_zeros=False)
degrees_nonzero = g.degrees_array(remove_zeros=True)


## 9. Métricas básicas da rede

Nesta etapa, medimos:

- `|V|`: ordem do grafo;
- `|E|`: tamanho do grafo;
- densidade;
- grau médio;
- clustering médio;
- estatísticas descritivas dos graus.

Essas métricas ajudam a entender o quão esparsa ou concentrada é a rede.


In [ ]:
print_basic_metrics(g)

### Leitura esperada desses resultados

Na execução atual, os principais valores observados são:

- `|V| = 36692`
- `|E| = 183831`
- `densidade = 0.00027310`
- `grau médio = 10.0202`
- `clustering médio = 0.496983`

Isso sugere:
- uma rede muito esparsa em termos globais;
- conectividade média relativamente baixa quando comparada ao número total possível de conexões;
- presença de agrupamentos locais relevantes, refletidos no clustering médio.


## 10. Inspeção manual de vértices

A inspeção manual ajuda a explicar, na prática, como alguns vértices têm comportamento muito diferente de outros.

Em redes reais, isso costuma ser útil para mostrar:
- vértices de grau muito baixo;
- vértices com papel mais central;
- diferenças locais de clustering.


In [ ]:
inspect_vertices(g, [0, 1, 2, 3, 10])

## 11. Gráfico 1 — distribuição de graus em escala linear

Esse gráfico mostra `P(k)` de forma direta.

Em geral, ele evidencia que:
- graus pequenos concentram a maior parte da massa;
- a frequência cai rapidamente conforme o grau cresce.


In [ ]:
plot_degree_distribution_linear(
    degrees_nonzero,
    "Distribuição de Graus P(k) em Escala Linear - Enron",
)

## 12. Gráfico 2 — distribuição de graus em escala log-log

Esse gráfico é importante porque distribuições com cauda pesada costumam ficar mais interpretáveis em escala log-log.

Se os pontos formam uma tendência aproximadamente linear em parte da cauda, isso pode ser compatível com power law — mas não serve, sozinho, como prova definitiva.


In [ ]:
plot_loglog(
    degrees_nonzero,
    "Distribuição de Graus P(k) em Escala Log-Log - Enron",
)

## 13. Ajuste por lei de potência

Agora ajustamos formalmente a distribuição usando a biblioteca `powerlaw`.


In [ ]:
fit, gamma, xmin, sigma, ks, n_tail = fit_powerlaw(degrees_nonzero)

print("=" * 60)
print("AJUSTE DE LEI DE POTÊNCIA")
print("=" * 60)
print(f"gamma = {gamma:.4f}")
print(f"xmin = {xmin}")
print(f"sigma = {sigma:.4f}")
print(f"KS = {ks:.4f}")
print(f"n_cauda = {n_tail}")
print()


### Interpretação dos parâmetros obtidos

Na execução atual, foram reportados:

- `gamma = 1.9725`
- `xmin = 4.0`
- `sigma = 0.0076`
- `KS = 0.0155`
- `n_cauda = 16514`

Esses valores mostram que:
- a análise por power law foi aplicada a partir de `xmin = 4`;
- a cauda considerada é grande em quantidade de observações;
- o ajuste merece ser discutido com base nos parâmetros, e não apenas pela estética do gráfico.


## 14. Gráfico 3 — distribuição log-log com ajuste visual

Esse gráfico combina:
- os pontos observados da distribuição;
- a linha do ajuste visual de power law;
- a marcação de `xmin`.

Ele ajuda bastante na apresentação, porque mostra visualmente onde começa a cauda analisada.


In [ ]:
plot_loglog_with_fit(
    degrees_nonzero,
    fit,
    ks,
    n_tail,
    "Ajuste de Lei de Potência - Distribuição de Graus Enron",
)

## 15. Discussão crítica sobre escala livre

Aqui está o ponto mais importante da interpretação final.

A atividade não pede que a equipe “force” uma conclusão. Pelo contrário: parte da nota está justamente em discutir os limites da classificação.

Por isso, a leitura adotada neste notebook é:
- existe evidência de **cauda pesada**;
- a cauda da distribuição pode ser analisada com **ajuste por lei de potência**;
- isso **não implica automaticamente** que toda a rede seja perfeitamente escala livre;
- a conclusão mais adequada é que a rede apresenta **comportamento compatível com power law na cauda**, devendo a classificação global ser feita com cautela.


In [ ]:
print_scale_free_conclusion(gamma, xmin, ks, n_tail)

## 16. Comparação conceitual com outras categorias

Mesmo sem executar uma segunda base neste notebook, vale registrar uma comparação conceitual importante para a apresentação:

- **redes de comunicação**, como a Enron, tendem a permitir a formação de hubs;
- **redes de estradas** costumam ter graus mais limitados por restrições espaciais e físicas;
- portanto, é esperado que a Enron apresente distribuição de graus mais heterogênea do que redes viárias, por exemplo.

Esse contraste ajuda a conectar os resultados empíricos com a interpretação por domínio.


## 17. Conclusão final

A análise da rede **Enron Email Network** mostrou que:

- a modelagem como **grafo não direcionado e não ponderado** é consistente com a descrição do dataset;
- o tratamento de duplicatas invertidas foi essencial para obter `|E|` compatível com a base oficial;
- a rede apresenta:
  - `|V| = 36692`
  - `|E| = 183831`
  - densidade muito baixa
  - grau médio de aproximadamente `10.02`
  - clustering médio de aproximadamente `0.497`
- a distribuição de graus possui forte assimetria e cauda pesada;
- há um ajuste plausível de **lei de potência na cauda**, com:
  - `gamma = 1.9725`
  - `xmin = 4.0`
  - `KS = 0.0155`
  - `n_cauda = 16514`

### Fechamento
A melhor conclusão metodológica é:

> a rede **não deve ser rotulada de forma automática** como perfeitamente escala livre, mas apresenta **comportamento compatível com power law na cauda da distribuição de graus**, o que é coerente com redes reais de comunicação.
